# 02. Source confounding and leave-one-source-out

## Setup

In [ ]:
from pathlib import Path
import json
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").is_file():
    if root == root.parent:
        raise RuntimeError("Open this notebook inside the project repository")
    root = root.parent
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))
if str(root) not in sys.path:
    sys.path.insert(0, str(root))  # model_aza

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from model_aza.training import transfer, tuning
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)


def frame(records):
    table = pd.DataFrame(records)
    for column in ("max_depth", "max_samples"):
        if column in table:
            table[column] = table[column].astype(str)
    return table


def show_and_save(figure, run, name):
    figure.tight_layout()
    figure.savefig(run / f"{name}.png", dpi=150)
    display(figure)
    plt.close(figure)

## Inputs

In [ ]:
development_path = ""  # Optional features_development.json; blank uses preprocessing 03
cohorts = list(transfer.FULL_COHORTS)
features_path, development = tuning.load_development(root, development_path or None)
print("Development features:", features_path)

In [ ]:
spatial_selection_run = ""  # Optional pipeline_outputs/transfer/spatial_selection_<id>; blank uses the latest
spatial_selection_run = Path(spatial_selection_run) if spatial_selection_run else tuning.latest_run(root, "spatial_selection", area=transfer.AREA)
parameters = json.loads((spatial_selection_run / "selected.json").read_text())
print("spatial_selection run:", spatial_selection_run)
parameters

## Crop by source parcel counts

In [ ]:
run = tuning.new_run(root, "source_holdout", features_path, area=transfer.AREA,
                     spatial_selection_run=str(spatial_selection_run))
for cohort in cohorts:
    table = transfer.confounding_table(transfer.development_rows(development, cohort))
    tuning.save_records(run, f"confounding_{cohort}", table)
    print(cohort)
    display(pd.DataFrame(table))

## Crop by site cell parcel counts

In [ ]:
for cohort in cohorts:
    rows = transfer.development_rows(development, cohort)
    parcels = {r["partition_group"]: (r["crop"], r["site_cell"]) for r in rows}
    table = pd.Series(list(parcels.values())).value_counts().rename_axis("crop_site").reset_index(name="parcels")
    table[["crop", "site_cell"]] = pd.DataFrame(table.crop_site.tolist(), index=table.index)
    print(cohort, "site cells per crop:", table.groupby("crop").site_cell.nunique().to_dict())
    display(table.pivot_table(index="site_cell", columns="crop", values="parcels", fill_value=0)
            .sort_values(list(transfer.CROPS), ascending=False).head(15))

## Source predictability within crop

In [ ]:
predictability = []
for cohort in cohorts:
    rows = transfer.development_rows(development, cohort)
    for record in transfer.source_predictability(rows):
        predictability.append({"cohort": cohort, **record})
tuning.save_records(run, "source_predictability", predictability)
display(pd.DataFrame(predictability).round(3))

## Leave-one-source-out

In [ ]:
holdout = []
for cohort in cohorts:
    rows = transfer.development_rows(development, cohort)
    for role, params in (("current", tuning.BASELINE), ("spatial_selected", parameters[cohort])):
        for record in transfer.leave_one_source_out(rows, params):
            holdout.append({"cohort": cohort, "role": role, **record})
        print(cohort, role, "done", flush=True)
tuning.save_records(run, "leave_one_source_out", holdout)
display(pd.DataFrame(holdout).round(3))

## Figure

In [ ]:
scored = pd.DataFrame([r for r in holdout if r["status"] == "scored"])
recall_columns = [c for c in scored.columns if c.endswith("_recall")]
figure, axes = plt.subplots(1, len(cohorts), figsize=(7 * len(cohorts), 4), sharey=True)
for axis, cohort in zip(axes, cohorts):
    table = scored[(scored.cohort == cohort) & (scored.role == "current")].set_index("held_out_source")[recall_columns]
    table.plot(kind="bar", ax=axis)
    axis.set_title(f"{cohort} (current settings)")
    axis.set_ylabel("Parcel recall on held-out source")
    axis.set_ylim(0, 1)
show_and_save(figure, run, "leave_one_source_out")
print("Run folder:", run)